# 🚀 FaceKey Collector — Kaggle
### Dedicated Worker: `kaggle-worker-1` (3 Parallel Workers, 1 Face Only, Turbo 150+ FPS)
- **Queue**: `youtube_urls_kaggle.txt` (117 Dedicated Videos — Zero Overlap)
- **Storage**: Packages 5-session `.tar.gz` chunks into `/kaggle/working/FaceKeyDataset/chunks/`.
- **Local PC Data Used**: **0 MB**.

In [ ]:
# ==============================================================================
# ⚡ 1-CLICK INSTANT RUNNER: KAGGLE WORKER
# ==============================================================================
import os, sys, shutil

# 1. Working directories
DRIVE_DIR = "/kaggle/working/FaceKeyDataset"
for folder in ("chunks", "checkpoints", "locks", "data"):
    os.makedirs(f"{DRIVE_DIR}/{folder}", exist_ok=True)

# 2. Update to latest GitHub repository code (Self-Healing)
%cd /kaggle/working
APP_DIR = "/kaggle/working/FaceKeyAnimation"
if os.path.isdir(os.path.join(APP_DIR, ".git")):
    print("[*] Updating FaceKeyAnimation to latest GitHub main...")
    !cd {APP_DIR} && git fetch --all --prune && git reset --hard origin/main && git clean -fd
else:
    print("[*] Fresh clone of FaceKeyAnimation from GitHub...")
    !rm -rf {APP_DIR}
    !git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}

# 3. Fast Cloud Dependencies & Deno JS solver
!pip install -q -U --pre "yt-dlp[default]" mediapipe opencv-python-headless numpy sounddevice mss av scipy
!if ! command -v deno &> /dev/null; then curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 && ln -sf /root/.deno/bin/deno /usr/local/bin/deno; fi

# 4. Dedicated Queue for Kaggle (Zero Overlap with Colab & Lightning)
urls_file = os.path.join(APP_DIR, "sessions", "youtube_urls_kaggle.txt")

# 5. Launch Collector (3 Workers across CPU cores, 1 Face Only, Turbo Mode)
print("\n[*] Launching FaceKey Turbo Multi-Worker Kaggle Collector (kaggle-worker-1)...")
!python scripts/cloud_data_collector.py \
    --drive-dir "{DRIVE_DIR}" \
    --worker-id "kaggle-worker-1" \
    --urls-file "{urls_file}" \
    --chunk-size 5 \
    --quality 480p \
    --num-workers 3


---
# 📦 KAGGLE STORAGE MANAGEMENT & DATA TRANSFER (0 MB LOCAL DATA)
> [!IMPORTANT]
> Kaggle is an isolated cloud environment with ~20 GB disk space. To transfer your collected chunks into your **5 TB Google Drive (`11VbtMpmNATsrBZxkPLpA2gPTtgaRFNlA`)** using **0 MB of your home internet**, choose **Option 1 (Fastest & Recommended)** or **Option 2** below.

In [ ]:
# 📊 Step 1: Check Kaggle Disk Space & Pending Chunks
%cd /kaggle/working/FaceKeyAnimation
!python scripts/cloud_drive_uploader.py --method status


---
### 🌟 OPTION 1 (RECOMMENDED): Hugging Face Cloud Bridge (1 Minute Setup, 0 GCP Config)
> [!TIP]
> **Why this is the easiest method:**
> 1. Sign up for free at [huggingface.co](https://huggingface.co) (takes 30 seconds).
> 2. Create a Write Token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (Role: `Write`).
> 3. Paste your token and repo name below (e.g. `your_username/facekey-chunks`).
> 4. Run the cell: It uploads all `.tar.gz` chunks directly to a free, private dataset and frees up Kaggle disk space.
> 5. Then in **Colab (`1_Colab_Data_Collector.ipynb`)**, run the final cell to pull all chunks directly into your 5 TB Google Drive at 100+ MB/s!

In [ ]:
# 🚀 Upload Chunks to Hugging Face Private Dataset (Cloud-to-Cloud, 0 MB Local Data)
HF_TOKEN = "YOUR_HF_WRITE_TOKEN"
HF_REPO = "YOUR_USERNAME/facekey-chunks"

!python scripts/cloud_drive_uploader.py \
    --method hf \
    --hf-token "{HF_TOKEN}" \
    --hf-repo "{HF_REPO}"


---
### ☁️ OPTION 2: Direct Google Drive Upload via Google Cloud Service Account
> [!WARNING]
> **Requires a Google Cloud IAM Service Account JSON key!**
> If you have NOT created a service account, **do NOT run the cell below** — use **Option 1** above instead.
>
> **To set up a Service Account:**
> 1. Go to [Google Cloud Console](https://console.cloud.google.com) and enable the **Google Drive API**.
> 2. Under **IAM & Admin** -> **Service Accounts**, click **Create Service Account**.
> 3. Click your service account -> **Keys** -> **Add Key** -> **Create new key** -> **JSON**.
> 4. In Google Drive, share your folder (`11VbtMpmNATsrBZxkPLpA2gPTtgaRFNlA`) with the service account email as **Editor**.
> 5. Upload that downloaded `.json` file to Kaggle as `/kaggle/working/service_account.json`.

In [ ]:
# ☁️ Direct Cloud-to-Cloud Upload to Google Drive (Requires /kaggle/working/service_account.json)
SERVICE_ACCOUNT_JSON = "/kaggle/working/service_account.json"
GOOGLE_DRIVE_FOLDER_ID = "11VbtMpmNATsrBZxkPLpA2gPTtgaRFNlA"

!python scripts/cloud_drive_uploader.py \
    --method gdrive \
    --service-account "{SERVICE_ACCOUNT_JSON}" \
    --folder-id "{GOOGLE_DRIVE_FOLDER_ID}"
